# Admissibility Vacuum Diagnostics (Toy Developmental Model)

This notebook simulates a multi-component developmental system and computes three diagnostics for an admissibility vacuum:
- Effective dimension (spectral entropy proxy)
- Predictive invalidation over time
- Contradiction index across component viability judgments

Dependencies: `numpy`, `matplotlib`, `scikit-learn`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [ ]:
# ---------- Parameters ----------
SEED = 42
rng = np.random.default_rng(SEED)

N = 4000           # number of simulated individuals
T = 60             # timesteps (gestational units)
k = 3              # number of developmental components (e.g., lungs, heart, brain)

# baseline params: component sensitivities and thresholds
component_weights = np.array([1.0, 0.9, 0.8])   # genotype scaling per organ
thresholds = np.array([0.9, 0.85, 0.8])         # per-organ viability thresholds at birth
base_growth = 0.015
insult_prob = 0.03
insult_scale = 0.12

# Gate: late collapse of maternal support around t_gate
t_gate = int(0.75 * T)
gate_severity = 0.5    # multiplies maternal support after gate (1.0 = no change)

# prenatal_boost is akin to protecting p (Carrier support)
prenatal_boost = 0.008  # additive per-step boost

In [ ]:
def sample_features(n):
    genotype = rng.uniform(0.4, 1.0, size=n)
    epigenetic = np.clip(rng.normal(0.5, 0.16, size=n), 0.0, 1.0)
    maternal_health = rng.uniform(0.3, 1.0, size=n)
    return genotype, epigenetic, maternal_health


def simulate(N, T, k, prenatal_boost=0.0):
    genotype, epigenetic, maternal = sample_features(N)
    sens = genotype[:, None] * (epigenetic[:, None] * component_weights[None, :])

    states = np.zeros((N, T + 1, k))
    states[:, 0, :] = 0.06 + 0.02 * rng.normal(size=(N, k))

    maternal_support = maternal.copy()
    for t in range(1, T + 1):
        if t >= t_gate:
            maternal_support = maternal_support * (
                1.0 - gate_severity * (t - t_gate + 1) / (T - t_gate + 1)
            )
            maternal_support = np.clip(maternal_support, 0.0, 1.0)

        growth = base_growth * sens * maternal_support[:, None]
        growth += prenatal_boost * maternal_support[:, None]

        insults = (rng.random((N, k)) < insult_prob).astype(float)
        insults *= rng.exponential(scale=insult_scale, size=(N, k))
        insults *= (1.0 - 0.5 * maternal_support)[:, None]

        states[:, t, :] = np.clip(states[:, t - 1, :] + growth - insults, 0.0, 2.0)

    return states, genotype, epigenetic, maternal


def effective_dim(M):
    M0 = M - M.mean(axis=0, keepdims=True)
    cov = np.cov(M0, rowvar=False)
    s = np.linalg.eigvalsh(cov)
    s = np.clip(s, 0.0, None)

    total = s.sum()
    if total <= 0:
        return 0.0

    p = s / total
    H = -np.sum([pi * np.log(pi) for pi in p if pi > 0.0])
    return float(np.exp(H))


def contradiction_index(M, thresholds):
    ok = M >= thresholds[None, :]
    has_viable = ok.any(axis=1)
    has_not_viable = (~ok).any(axis=1)
    contradictory = has_viable & has_not_viable
    return contradictory.mean()


def predictive_error_over_time(states, thresholds):
    N, T1, k = states.shape
    final_state = states[:, -1, :]
    survive = (final_state >= thresholds[None, :]).all(axis=1).astype(int)

    errors = np.zeros(T1)
    for t in range(T1):
        X = states[:, t, :]
        clf = RandomForestClassifier(n_estimators=60, random_state=SEED, n_jobs=-1)
        try:
            scores = cross_val_score(clf, X, survive, cv=3, scoring='accuracy')
            err = 1.0 - scores.mean()
        except Exception:
            maj = np.bincount(survive).argmax()
            err = 1.0 - (survive == maj).mean()
        errors[t] = err

    return errors, survive

In [ ]:
# ---------- Run simulation ----------
states, genotype, epi, maternal = simulate(N, T, k, prenatal_boost=prenatal_boost)
Nsim, T1, k = states.shape

eff_dims = np.zeros(T1)
contrad = np.zeros(T1)
for t in range(T1):
    M = states[:, t, :]
    eff_dims[t] = effective_dim(M)
    contrad[t] = contradiction_index(M, thresholds)

pred_errs, survive = predictive_error_over_time(states, thresholds)

In [ ]:
# ---------- Plot results ----------
times = np.arange(T1)

plt.figure(figsize=(10, 6))
ax1 = plt.subplot(311)
ax1.plot(times, eff_dims, label='Effective dimension (exp spectral entropy)', color='C0')
ax1.axvline(t_gate, color='k', linestyle='--', alpha=0.6, label='Gate onset')
ax1.set_ylabel('Effective dim')
ax1.legend(loc='best')

ax2 = plt.subplot(312, sharex=ax1)
ax2.plot(times, pred_errs, label='Predictive error (1 - CV accuracy)', color='C1')
ax2.axvline(t_gate, color='k', linestyle='--', alpha=0.6)
ax2.set_ylabel('Predictive error')
ax2.legend(loc='best')

ax3 = plt.subplot(313, sharex=ax1)
ax3.plot(times, contrad, label='Contradiction index (conflicting organ verdicts)', color='C2')
ax3.axvline(t_gate, color='k', linestyle='--', alpha=0.6)
ax3.set_xlabel('Time')
ax3.set_ylabel('Contradiction fraction')
ax3.legend(loc='best')

plt.tight_layout()
plt.suptitle('Admissibility vacuum diagnostics (toy developmental model)', y=1.02)
plt.show()

In [ ]:
# ---------- Vacuum interval summary ----------
drop_idx = np.argmin(eff_dims[t_gate:]) + t_gate
peak_err_idx = np.argmax(pred_errs[t_gate:]) + t_gate
peak_contrad_idx = np.argmax(contrad[t_gate:]) + t_gate

print('Gate onset t_gate =', t_gate)
print('Min effective_dim after gate at t =', drop_idx, 'value =', eff_dims[drop_idx])
print('Max predictive error after gate at t =', peak_err_idx, 'value =', pred_errs[peak_err_idx])
print('Max contradiction index after gate at t =', peak_contrad_idx, 'value =', contrad[peak_contrad_idx])

# Optional: save figure
# plt.savefig('adm_vacuum_demo.png', bbox_inches='tight', dpi=200)

## Visual Signature of an Admissibility Vacuum

- **Effective dimension**: if it drops sharply after `t_gate`, the ensemble collapses onto a lower-dimensional manifold (loss of independent degrees of freedom), indicating contraction of admissibility.
- **Predictive error**: if error rises near/after the gate, predictors trained in an earlier regime lose validity for successor outcomes, signaling operational admissibility failure.
- **Contradiction index**: a transient peak means local predicates (organ viability tests) disagree more often, indicating inconsistent coordination between continuation and coherence constraints.
- If all three show a transient degradation interval, that interval is the empirical admissibility vacuum.

## Mapping to GRF Language

- States $x(t)$ represent jurisdictional microstates.
- `t_gate` models a regime change where carrier support weakens.
- Effective dimension proxies $\mathrm{Coord}(\Lambda, \Phi, t)$: lower values mean fewer coordinated degrees of freedom; rapid shifts indicate collapse.
- Predictive error operationalizes changes in $\mathrm{Adm}(J,p,t)$: when old predictors fail, admissibility has changed.
- Contradiction index captures predicate disagreement in the vacuum interval.

## Extensions and Experiments

- Sweep `N`, `gate_severity`, `prenatal_boost`, and `insult_prob`; generate heatmaps of vacuum depth and duration.
- Replace RandomForest with logistic regression or trajectory-aware models using flattened prefix histories up to time $t$.
- Build a decoherence toy with qubit Bloch vectors and a gate-time decoherence channel; track purity and predictability collapse.
- Reuse the pregnancy model and compute the same three proxies for cross-model comparison.

## Limitations and Cautions

- This is a toy model; dynamics, thresholds, and dimensionality are illustrative.
- Effective dimension depends on ensemble variance and $k$; interpret relatively (before vs after gate), not absolutely.
- Predictive error depends on model class and feature choice; expanding the feature arena can restore predictability, which operationally corresponds to enlarging jurisdiction.

## Parameter Sweep: Vacuum Depth and Duration Heatmaps

This section sweeps key parameters and estimates:
- **Vacuum depth**: normalized post-gate degradation across effective dimension, predictive error, and contradiction
- **Vacuum duration**: number of post-gate timepoints flagged as degraded coordination

To keep runtime practical, this sweep uses a reduced population and sampled timesteps for predictive error.

In [ ]:
# Fast scenario runner for parameter sweeps
# Uses reduced N and sampled timesteps to keep runtime practical.

def simulate_with_params(
    N_local,
    T_local,
    k_local,
    gate_severity_local,
    prenatal_boost_local,
    insult_prob_local,
    seed_offset=0,
):
    rng_local = np.random.default_rng(SEED + seed_offset)

    genotype = rng_local.uniform(0.4, 1.0, size=N_local)
    epigenetic = np.clip(rng_local.normal(0.5, 0.16, size=N_local), 0.0, 1.0)
    maternal = rng_local.uniform(0.3, 1.0, size=N_local)

    sens = genotype[:, None] * (epigenetic[:, None] * component_weights[None, :])
    states_local = np.zeros((N_local, T_local + 1, k_local))
    states_local[:, 0, :] = 0.06 + 0.02 * rng_local.normal(size=(N_local, k_local))

    t_gate_local = int(0.75 * T_local)
    maternal_support = maternal.copy()

    for t in range(1, T_local + 1):
        if t >= t_gate_local:
            maternal_support = maternal_support * (
                1.0 - gate_severity_local * (t - t_gate_local + 1) / (T_local - t_gate_local + 1)
            )
            maternal_support = np.clip(maternal_support, 0.0, 1.0)

        growth = base_growth * sens * maternal_support[:, None]
        growth += prenatal_boost_local * maternal_support[:, None]

        insults = (rng_local.random((N_local, k_local)) < insult_prob_local).astype(float)
        insults *= rng_local.exponential(scale=insult_scale, size=(N_local, k_local))
        insults *= (1.0 - 0.5 * maternal_support)[:, None]

        states_local[:, t, :] = np.clip(states_local[:, t - 1, :] + growth - insults, 0.0, 2.0)

    return states_local


def predictive_error_sampled(states_local, thresholds_local, times_idx, seed_offset=0):
    final_state = states_local[:, -1, :]
    survive = (final_state >= thresholds_local[None, :]).all(axis=1).astype(int)

    sampled_errors = np.zeros(len(times_idx))
    for i, t in enumerate(times_idx):
        X = states_local[:, t, :]
        clf = RandomForestClassifier(n_estimators=25, random_state=SEED + seed_offset, n_jobs=-1)
        try:
            scores = cross_val_score(clf, X, survive, cv=2, scoring='accuracy')
            sampled_errors[i] = 1.0 - scores.mean()
        except Exception:
            maj = np.bincount(survive).argmax()
            sampled_errors[i] = 1.0 - (survive == maj).mean()

    return sampled_errors


def vacuum_metrics_for_scenario(
    gate_severity_local,
    prenatal_boost_local,
    insult_prob_local,
    N_local=1200,
    T_local=T,
    k_local=k,
    seed_offset=0,
):
    states_local = simulate_with_params(
        N_local,
        T_local,
        k_local,
        gate_severity_local,
        prenatal_boost_local,
        insult_prob_local,
        seed_offset=seed_offset,
    )

    t_gate_local = int(0.75 * T_local)
    T1_local = T_local + 1

    # Sample timesteps for speed.
    times_idx = np.unique(np.concatenate([
        np.arange(0, T1_local, 4),
        np.array([t_gate_local, T1_local - 1]),
    ])).astype(int)

    eff_s = np.zeros(len(times_idx))
    con_s = np.zeros(len(times_idx))
    for i, t in enumerate(times_idx):
        M = states_local[:, t, :]
        eff_s[i] = effective_dim(M)
        con_s[i] = contradiction_index(M, thresholds)

    pred_s = predictive_error_sampled(states_local, thresholds, times_idx, seed_offset=seed_offset)

    pre_mask = times_idx < t_gate_local
    post_mask = times_idx >= t_gate_local

    eff_pre, eff_post = eff_s[pre_mask], eff_s[post_mask]
    pred_pre, pred_post = pred_s[pre_mask], pred_s[post_mask]
    con_pre, con_post = con_s[pre_mask], con_s[post_mask]

    eps = 1e-9
    eff_mu, eff_sd = eff_pre.mean(), eff_pre.std() + eps
    pred_mu, pred_sd = pred_pre.mean(), pred_pre.std() + eps
    con_mu, con_sd = con_pre.mean(), con_pre.std() + eps

    eff_drop = np.maximum(0.0, (eff_mu - eff_post) / eff_sd)
    pred_rise = np.maximum(0.0, (pred_post - pred_mu) / pred_sd)
    con_rise = np.maximum(0.0, (con_post - con_mu) / con_sd)

    # Depth aggregates normalized post-gate degradations.
    depth = float(np.mean(eff_drop + pred_rise + con_rise))

    # Duration counts post-gate sampled timepoints where at least 2/3 criteria degrade.
    flags = (
        ((eff_post < (eff_mu - 0.5 * eff_sd)).astype(int))
        + ((pred_post > (pred_mu + 0.5 * pred_sd)).astype(int))
        + ((con_post > (con_mu + 0.5 * con_sd)).astype(int))
    ) >= 2
    duration = int(flags.sum())

    return depth, duration


def sweep_heatmap(x_vals, y_vals, scenario_fn):
    depth_map = np.zeros((len(y_vals), len(x_vals)))
    duration_map = np.zeros((len(y_vals), len(x_vals)))
    scenario_id = 0

    for yi, y in enumerate(y_vals):
        for xi, x in enumerate(x_vals):
            scenario_id += 1
            d, dur = scenario_fn(x, y, scenario_id)
            depth_map[yi, xi] = d
            duration_map[yi, xi] = dur

    return depth_map, duration_map


# Grids
sev_vals = np.linspace(0.1, 0.9, 6)
boost_vals = np.linspace(0.0, 0.016, 6)
insult_vals = np.linspace(0.01, 0.07, 6)

# 1) gate_severity x prenatal_boost, fixed insult_prob
depth_sb, dur_sb = sweep_heatmap(
    sev_vals,
    boost_vals,
    lambda sev, boost, sid: vacuum_metrics_for_scenario(
        gate_severity_local=sev,
        prenatal_boost_local=boost,
        insult_prob_local=insult_prob,
        seed_offset=10_000 + sid,
    ),
)

# 2) gate_severity x insult_prob, fixed prenatal_boost
depth_si, dur_si = sweep_heatmap(
    sev_vals,
    insult_vals,
    lambda sev, ins, sid: vacuum_metrics_for_scenario(
        gate_severity_local=sev,
        prenatal_boost_local=prenatal_boost,
        insult_prob_local=ins,
        seed_offset=20_000 + sid,
    ),
)

# 3) prenatal_boost x insult_prob, fixed gate_severity
depth_bi, dur_bi = sweep_heatmap(
    boost_vals,
    insult_vals,
    lambda boost, ins, sid: vacuum_metrics_for_scenario(
        gate_severity_local=gate_severity,
        prenatal_boost_local=boost,
        insult_prob_local=ins,
        seed_offset=30_000 + sid,
    ),
)


def plot_heat(ax, Z, x_vals, y_vals, xlab, ylab, title, cmap='viridis'):
    im = ax.imshow(
        Z,
        aspect='auto',
        origin='lower',
        extent=[x_vals.min(), x_vals.max(), y_vals.min(), y_vals.max()],
        cmap=cmap,
    )
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


plt.figure(figsize=(14, 8))
ax = plt.subplot(231)
plot_heat(ax, depth_sb, sev_vals, boost_vals, 'gate_severity', 'prenatal_boost', 'Depth: severity x boost')
ax = plt.subplot(232)
plot_heat(ax, depth_si, sev_vals, insult_vals, 'gate_severity', 'insult_prob', 'Depth: severity x insult_prob')
ax = plt.subplot(233)
plot_heat(ax, depth_bi, boost_vals, insult_vals, 'prenatal_boost', 'insult_prob', 'Depth: boost x insult_prob')

ax = plt.subplot(234)
plot_heat(ax, dur_sb, sev_vals, boost_vals, 'gate_severity', 'prenatal_boost', 'Duration: severity x boost', cmap='magma')
ax = plt.subplot(235)
plot_heat(ax, dur_si, sev_vals, insult_vals, 'gate_severity', 'insult_prob', 'Duration: severity x insult_prob', cmap='magma')
ax = plt.subplot(236)
plot_heat(ax, dur_bi, boost_vals, insult_vals, 'prenatal_boost', 'insult_prob', 'Duration: boost x insult_prob', cmap='magma')

plt.tight_layout()
plt.show()

print('Sweep complete.')
print('Interpretation hint: lower depth/duration regions indicate weaker or shorter admissibility vacuum signatures.')

## Auto Summary of Sweep Extremes

This cell extracts the strongest and weakest admissibility-vacuum signatures from the three parameter grids.
It reports top regions for both vacuum depth and vacuum duration.

In [ ]:
def top_k_regions(Z, x_vals, y_vals, x_name, y_name, k_top=5, highest=True):
    flat = Z.ravel()
    order = np.argsort(flat)
    idx = order[-k_top:][::-1] if highest else order[:k_top]

    rows = []
    ncols = Z.shape[1]
    for rank, pos in enumerate(idx, start=1):
        yi = pos // ncols
        xi = pos % ncols
        rows.append({
            'rank': rank,
            x_name: float(x_vals[xi]),
            y_name: float(y_vals[yi]),
            'value': float(Z[yi, xi]),
        })
    return rows


def print_region_table(title, rows, x_name, y_name, value_label):
    print(title)
    print(f"{'#':>2}  {x_name:>14}  {y_name:>14}  {value_label:>12}")
    for r in rows:
        print(f"{r['rank']:>2}  {r[x_name]:>14.6f}  {r[y_name]:>14.6f}  {r['value']:>12.6f}")
    print()


# Depth extremes
print('\n=== Vacuum Depth Extremes ===\n')
print_region_table(
    'Lowest depth: severity x boost',
    top_k_regions(depth_sb, sev_vals, boost_vals, 'gate_severity', 'prenatal_boost', highest=False),
    'gate_severity',
    'prenatal_boost',
    'depth',
)
print_region_table(
    'Highest depth: severity x boost',
    top_k_regions(depth_sb, sev_vals, boost_vals, 'gate_severity', 'prenatal_boost', highest=True),
    'gate_severity',
    'prenatal_boost',
    'depth',
)

print_region_table(
    'Lowest depth: severity x insult_prob',
    top_k_regions(depth_si, sev_vals, insult_vals, 'gate_severity', 'insult_prob', highest=False),
    'gate_severity',
    'insult_prob',
    'depth',
)
print_region_table(
    'Highest depth: severity x insult_prob',
    top_k_regions(depth_si, sev_vals, insult_vals, 'gate_severity', 'insult_prob', highest=True),
    'gate_severity',
    'insult_prob',
    'depth',
)

print_region_table(
    'Lowest depth: prenatal_boost x insult_prob',
    top_k_regions(depth_bi, boost_vals, insult_vals, 'prenatal_boost', 'insult_prob', highest=False),
    'prenatal_boost',
    'insult_prob',
    'depth',
)
print_region_table(
    'Highest depth: prenatal_boost x insult_prob',
    top_k_regions(depth_bi, boost_vals, insult_vals, 'prenatal_boost', 'insult_prob', highest=True),
    'prenatal_boost',
    'insult_prob',
    'depth',
)

# Duration extremes
print('\n=== Vacuum Duration Extremes ===\n')
print_region_table(
    'Lowest duration: severity x boost',
    top_k_regions(dur_sb, sev_vals, boost_vals, 'gate_severity', 'prenatal_boost', highest=False),
    'gate_severity',
    'prenatal_boost',
    'duration',
)
print_region_table(
    'Highest duration: severity x boost',
    top_k_regions(dur_sb, sev_vals, boost_vals, 'gate_severity', 'prenatal_boost', highest=True),
    'gate_severity',
    'prenatal_boost',
    'duration',
)

print_region_table(
    'Lowest duration: severity x insult_prob',
    top_k_regions(dur_si, sev_vals, insult_vals, 'gate_severity', 'insult_prob', highest=False),
    'gate_severity',
    'insult_prob',
    'duration',
)
print_region_table(
    'Highest duration: severity x insult_prob',
    top_k_regions(dur_si, sev_vals, insult_vals, 'gate_severity', 'insult_prob', highest=True),
    'gate_severity',
    'insult_prob',
    'duration',
)

print_region_table(
    'Lowest duration: prenatal_boost x insult_prob',
    top_k_regions(dur_bi, boost_vals, insult_vals, 'prenatal_boost', 'insult_prob', highest=False),
    'prenatal_boost',
    'insult_prob',
    'duration',
)
print_region_table(
    'Highest duration: prenatal_boost x insult_prob',
    top_k_regions(dur_bi, boost_vals, insult_vals, 'prenatal_boost', 'insult_prob', highest=True),
    'prenatal_boost',
    'insult_prob',
    'duration',
)

print('Summary extraction complete.')